# Paloma Humans Datasets

Create:
- Make European-only human HA-only pandemic swine dataset

Trees: Make 2 Fast Trees
- One with all pig sequences + human dataset
- One with only Paloma sequences + human dataset

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [8]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/"
downloads = home + "downloads/human_euro_N2_01-01-1976--12-31-2025/" 
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

complete_files = home + "complete_human/" 
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# if not os.path.exists(complete_files + "deduplicated/"): # checking if the directory exists or not
#     os.makedirs(complete_files + "deduplicated/") # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

In [9]:
# Get metadata and sequences

gisaid_metadata = []
gisaid_fastas = []
for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = fasta_df(file_name, states_ref)
            # fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
            # # All sequences should be human
            # fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2]) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
            # fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-5])
            # fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
            # fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
            # fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
            # fasta_df["Genotype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
            gisaid_fastas.append(fasta)
        else: # if ".xls" in file name
            metadata = pd.read_excel(file_name)
            gisaid_metadata.append(metadata)

# Concatenate metadata
metadata_concat = pd.DataFrame()
for metadata_file in gisaid_metadata:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

# Concatenate fastas
fasta_concat = pd.DataFrame()
for fasta in gisaid_fastas:
    fasta_concat = pd.concat([fasta_concat, fasta])

print(metadata_concat)
print(fasta_concat)


EPI_ISL_331848|A-Rostov-CRIE-1-2018|A_/_H3N2|NA|2018-09-05
EPI_ISL_331848|A-Rostov-CRIE-1-2018|A_/_H3N2|HA|2018-09-05
EPI_ISL_365464|A-Tambov-CRIE-304-2019|A_/_H3N2|HA|2019-03-01
EPI_ISL_308743|A-Tambov-CRIE-08-2017|A_/_H3N2|NA|2017-11-16
EPI_ISL_308743|A-Tambov-CRIE-08-2017|A_/_H3N2|HA|2017-11-16
EPI_ISL_308749|A-Kursk-CRIE-113-2018|A_/_H3N2|NA|2018-02-20
EPI_ISL_308749|A-Kursk-CRIE-113-2018|A_/_H3N2|HA|2018-02-20
EPI_ISL_308744|A-Volgograd-CRIE-30-2018|A_/_H3N2|NA|2018-01-22
EPI_ISL_308744|A-Volgograd-CRIE-30-2018|A_/_H3N2|HA|2018-01-22
EPI_ISL_308745|A-Volgograd-CRIE-32-2018|A_/_H3N2|NA|2018-01-15
EPI_ISL_308745|A-Volgograd-CRIE-32-2018|A_/_H3N2|HA|2018-01-15
EPI_ISL_308746|A-Voronezh-CRIE-84-2018|A_/_H3N2|NA|2018-02-12
EPI_ISL_308746|A-Voronezh-CRIE-84-2018|A_/_H3N2|HA|2018-02-12
EPI_ISL_308747|A-Orenburg-CRIE-102-2018|A_/_H3N2|NA|2018-02-14
EPI_ISL_308747|A-Orenburg-CRIE-102-2018|A_/_H3N2|HA|2018-02-14
EPI_ISL_19693062|80423825|A_/_H3N2|NA|2024-11-27
EPI_ISL_19693062|80423825|A_/_

## Make names

In [11]:
# Find "animals" (geographic locations indicating human sequence)

segment_fastas = []
unique_animals_all = []

unique_animals = sort_animals(fasta_concat) # Find unique animals
    # print("Animals: ", unique_animals)
unique_animals_all.append(unique_animals)

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv")

['crundale', 'toulon', 'enschede', 'holyhead', 'romania_bb', 'donetsk', 'podporozhye', 'shimsk', 'trefforest', 'nyzhninovgorod', 'argenteuil', 'poland', 'pontypridd', 'nimes', 'tomsk', 'aubenas', 'aberkenfig', 'buzau', 'foggia', 'saint-martin', 'brandenburg', 'coity', 'nitra', 'mountain_ash', 'conwy', 'slantsy', 'umea', 'karachay-cherkess', 'pais_vasco', 'bosnia_&_herzegovina', 'kursk', 'naryan-mar', 'barry', 'piestany', 'cyprus', 'human', 'cardiff', 'valday', 'wallonia', 'ilia.gr', 'eskilstuna', 'yaroslavl', 'joshkar-ola', 'mari_el', 'romania', 'estonia', 'ostersund', 'neath', 'picardie', 'catalonia', 'ukraine', 'yaroslavl_oblast', 'arad', 'cheboksary', 'baden-wurttemberg', 'austria1577520', 'elekmnar', 'novoaidar', 'korsakov', 'sweden', 'lisboa', 'drozhzhanoe', 'margam', 'jonkoping', 'marseille', 'severodonetsk', 'murmask', 'koygorodok', 'macon', 'navarra', 'st_clears', 'flint', 'ystrad_mynach', 'la-rochelle', 'volgograd_region', 'gatchina', 'south_australia', 'kerkira', 'athens.gr',

In [12]:
fasta = fix_animals(fasta_concat, animals_df) # Fix animals first

# >EPI_ID|Isolate_name|subtype|collection_date|host_type

xls = metadata_concat.rename(columns={"Isolate_Id":"Identifier"})

# Merge metadata with fasta
fasta_meta = fasta_concat.merge(xls, how="right", on="Identifier")

print(fasta_meta)
# Those with missing metadata get dropped
fasta_meta = fasta_meta.dropna(subset=["Identifier", "Isolate_Name_x", "Subtype_x", "Geo_Location", "Date Collected", "Host_Type"])
# Those with embargos get dropped
fasta_meta = fasta_meta[fasta_meta["Publishing_Embargo_Until"].isna()]

# Rename sequences 
new_name = ">" + fasta_meta["Identifier"] + "|" + fasta_meta["Isolate_Name_x"] + "|" + fasta_meta["Subtype_x"] + "|" + fasta_meta["Geo_Location"] + "|" + fasta_meta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_meta["Host_Type"] 
# print(fasta_seg["New_Name"])

fasta_meta["full_header"] = new_name

print(fasta_meta)


                                                   Header Isolate_Id  \
0       EPI_ISL_14846394|A/Norway/35950/2021|A_/_H3N2|...      35950   
1       EPI_ISL_14255999|A/Finland/266/2021|A_/_H3N2|H...        266   
2       EPI_ISL_14255999|A/Finland/266/2021|A_/_H3N2|N...        266   
3       EPI_ISL_14255773|A/Finland/265/2021|A_/_H3N2|N...        265   
4       EPI_ISL_14255773|A/Finland/265/2021|A_/_H3N2|H...        265   
...                                                   ...        ...   
420359  EPI_ISL_163508|A/Pays_de_Loire/1262/2014|A_/_H...       1262   
420360  EPI_ISL_163507|A/Pays_de_Loire/1266/2014|A_/_H...       1266   
420361  EPI_ISL_163506|A/Caen/349/2014|A_/_H3N2|HA|201...        349   
420362  EPI_ISL_163506|A/Caen/349/2014|A_/_H3N2|MP|201...        349   
420363  EPI_ISL_163506|A/Caen/349/2014|A_/_H3N2|NA|201...        349   

                   Isolate_Name_x Subtype_x Segment Location_Header  \
0             A/Norway/35950/2021      H3N2      HA          Nor

## Create FASTA

In [13]:
fasta_meta = fasta_meta.rename(columns={"Sequence":"sequence"})
df_to_fasta(fasta_meta, "human_euro_N2_01-01-1976--12-31-2025.fasta", complete_files)